# Lab 06 External V2 — 03 Fact Conditions

**Dataset:** Synthea Healthcare  
**Architecture:** External Delta tables  
**Compute:** Databricks Serverless compatible

## Purpose

Build `fact_conditions` at **one row per distinct patient-condition occurrence**,
resolve patient/date/condition/encounter keys, and persist the result as an external Delta table.

> Serverless compatibility rule: this notebook does **not** call
> `REFRESH TABLE`, `CACHE TABLE`, `UNCACHE TABLE`, or Spark cache-refresh APIs.


## 1. Runtime context

In [ ]:
def ensure_text_widget(name: str, default: str, label: str) -> None:
    try:
        dbutils.widgets.get(name)
    except Exception:
        dbutils.widgets.text(name, default, label)


def ensure_dropdown_widget(
    name: str,
    default: str,
    choices: list[str],
    label: str,
) -> None:
    try:
        dbutils.widgets.get(name)
    except Exception:
        dbutils.widgets.dropdown(name, default, choices, label)


ensure_text_widget("catalog", "dbr_dev", "01 Catalog")
ensure_text_widget("source_schema", "parvinbadalov", "02 Source schema")
ensure_text_widget(
    "source_volume_name",
    "lab06_gold_analytics",
    "03 Source volume",
)
ensure_text_widget(
    "target_schema",
    "parvinbadalov_lab06_ext",
    "04 Target schema",
)
ensure_text_widget(
    "external_gold_root",
    "REPLACE_WITH_EXTERNAL_GOLD_ROOT",
    "05 External Gold root",
)
ensure_dropdown_widget(
    "run_validation",
    "true",
    ["true", "false"],
    "06 Run validation",
)

catalog = dbutils.widgets.get("catalog").strip()
source_schema = dbutils.widgets.get("source_schema").strip()
source_volume_name = dbutils.widgets.get("source_volume_name").strip()
target_schema = dbutils.widgets.get("target_schema").strip()
external_gold_root = dbutils.widgets.get("external_gold_root").strip().rstrip("/")
run_validation = (
    dbutils.widgets.get("run_validation").strip().lower() == "true"
)

if not external_gold_root or external_gold_root == "REPLACE_WITH_EXTERNAL_GOLD_ROOT":
    raise ValueError(
        "external_gold_root must be passed by lab06_00_dev_runner "
        "or entered manually."
    )

if not external_gold_root.lower().startswith("abfss://"):
    raise ValueError("external_gold_root must be an abfss:// path.")

source_volume_path = (
    f"/Volumes/{catalog}/{source_schema}/{source_volume_name}"
)
source_csv_path = f"{source_volume_path}/source/csv"
reference_path = f"{source_volume_path}/reference"
target_schema_fqn = f"{catalog}.{target_schema}"

print(f"Catalog            : {catalog}")
print(f"Source schema      : {source_schema}")
print(f"Source volume      : {source_volume_name}")
print(f"Target schema      : {target_schema}")
print(f"External Gold root : {external_gold_root}")
print(f"Run validation     : {run_validation}")

## 2. Helpers and object names

In [ ]:
import sys
from pathlib import Path

from pyspark.sql import functions as F

current_dir = Path.cwd()
lab_root = current_dir.parent if current_dir.name == "notebooks" else current_dir

if str(lab_root) not in sys.path:
    sys.path.insert(0, str(lab_root))

from src.external_tables import (
    normalize_location,
    overwrite_external_delta,
    registered_table_location,
    register_external_delta_table,
    validate_registered_location,
)

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {target_schema_fqn}")

print("Serverless-safe external-table helpers loaded.")

TABLES = {
    "dim_date": f"{target_schema_fqn}.dim_date",
    "dim_patient": f"{target_schema_fqn}.dim_patient",
    "dim_condition": f"{target_schema_fqn}.dim_condition",
    "fact_encounters": f"{target_schema_fqn}.fact_encounters",
    "fact_conditions": f"{target_schema_fqn}.fact_conditions",
}

fact_location = f"{external_gold_root}/fact_conditions"
conditions_source = f"{reference_path}/conditions.csv"

for required in ["dim_date", "dim_patient", "dim_condition", "fact_encounters"]:
    if not spark.catalog.tableExists(TABLES[required]):
        raise RuntimeError(f"Missing required object: {TABLES[required]}")

print(f"Source : {conditions_source}")
print(f"Target : {TABLES['fact_conditions']}")
print(f"Path   : {fact_location}")

## 3. Load, validate, and prepare condition events

In [ ]:
conditions_src = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .csv(conditions_source)
)

required = {"START","STOP","PATIENT","ENCOUNTER","CODE","DESCRIPTION"}
missing = sorted(required - set(conditions_src.columns))
if missing:
    raise ValueError("conditions.csv missing: " + ", ".join(missing))

prepared_conditions = (
    conditions_src
    .select(
        F.to_date("START").alias("condition_start_date"),
        F.to_date("STOP").alias("condition_stop_date"),
        F.col("PATIENT").alias("patient_id"),
        F.col("ENCOUNTER").alias("encounter_id"),
        F.col("CODE").alias("condition_code"),
        F.col("DESCRIPTION").alias("condition_description"),
    )
    .dropDuplicates([
        "patient_id","encounter_id","condition_code","condition_description",
        "condition_start_date","condition_stop_date",
    ])
    .withColumn(
        "condition_event_key",
        F.xxhash64(
            "patient_id",
            "encounter_id",
            "condition_code",
            "condition_description",
            F.col("condition_start_date").cast("string"),
            F.coalesce(F.col("condition_stop_date").cast("string"), F.lit("")),
        ),
    )
)

grain = prepared_conditions.agg(
    F.count("*").alias("rows"),
    F.countDistinct("condition_event_key").alias("keys"),
    F.sum(F.when(F.col("condition_event_key").isNull(),1).otherwise(0)).alias("null_key"),
    F.sum(F.when(F.col("patient_id").isNull(),1).otherwise(0)).alias("null_patient"),
    F.sum(F.when(F.col("condition_code").isNull(),1).otherwise(0)).alias("null_code"),
    F.sum(F.when(F.col("condition_start_date").isNull(),1).otherwise(0)).alias("bad_start"),
).first()

source_failures = []
if grain["rows"] != grain["keys"]:
    source_failures.append("duplicate condition event")
for f in ["null_key","null_patient","null_code","bad_start"]:
    if (grain[f] or 0) > 0:
        source_failures.append(f)

if run_validation and source_failures:
    raise RuntimeError("Condition source validation failed: " + ", ".join(source_failures))

display(prepared_conditions.limit(10))

## 4. Resolve Gold keys

In [ ]:
dim_patient = spark.table(TABLES["dim_patient"]).select("patient_key","patient_id")
dim_condition = spark.table(TABLES["dim_condition"]).select(
    "condition_key","condition_code","condition_description"
)
dim_date = spark.table(TABLES["dim_date"]).select("date_key","full_date")
encounters = spark.table(TABLES["fact_encounters"]).select("encounter_key","encounter_id")

fact_conditions_df = (
    prepared_conditions.alias("c")
    .join(dim_patient.alias("p"), F.col("c.patient_id") == F.col("p.patient_id"), "left")
    .join(
        dim_condition.alias("dc"),
        (F.col("c.condition_code") == F.col("dc.condition_code"))
        & (F.col("c.condition_description") == F.col("dc.condition_description")),
        "left",
    )
    .join(dim_date.alias("d"), F.col("c.condition_start_date") == F.col("d.full_date"), "left")
    .join(encounters.alias("e"), F.col("c.encounter_id") == F.col("e.encounter_id"), "left")
    .select(
        F.col("c.condition_event_key"),
        F.col("p.patient_key"),
        F.col("dc.condition_key"),
        F.col("d.date_key").alias("condition_start_date_key"),
        F.col("e.encounter_key"),
        F.col("c.patient_id"),
        F.col("c.encounter_id"),
        F.col("c.condition_code"),
        F.col("c.condition_description"),
        F.col("c.condition_start_date"),
        F.col("c.condition_stop_date"),
        F.col("c.condition_stop_date").isNull().alias("is_active_condition"),
        F.when(
            F.col("c.condition_stop_date").isNotNull(),
            F.datediff("c.condition_stop_date", "c.condition_start_date"),
        ).otherwise(F.lit(None).cast("int")).alias("condition_duration_days"),
    )
)

display(fact_conditions_df.limit(10))

## 5. Validate foreign keys

In [ ]:
fk = fact_conditions_df.agg(
    F.sum(F.when(F.col("patient_key").isNull(),1).otherwise(0)).alias("patient"),
    F.sum(F.when(F.col("condition_key").isNull(),1).otherwise(0)).alias("condition"),
    F.sum(F.when(F.col("condition_start_date_key").isNull(),1).otherwise(0)).alias("date"),
    F.sum(
        F.when(
            F.col("encounter_id").isNotNull() & F.col("encounter_key").isNull(),
            1,
        ).otherwise(0)
    ).alias("encounter"),
).first()

fk_failures = [n for n in ["patient","condition","date","encounter"] if (fk[n] or 0) > 0]

display(spark.createDataFrame(
    [(n, int(fk[n] or 0), "PASS" if (fk[n] or 0) == 0 else "FAIL")
     for n in ["patient","condition","date","encounter"]],
    ["foreign_key","missing_rows","status"],
))

if run_validation and fk_failures:
    raise RuntimeError("fact_conditions FK validation failed: " + ", ".join(fk_failures))

## 6. Persist and reconcile

In [ ]:
overwrite_external_delta(
    spark,
    fact_conditions_df,
    TABLES["fact_conditions"],
    fact_location,
)

target = spark.table(TABLES["fact_conditions"])
target_profile = target.agg(
    F.count("*").alias("rows"),
    F.countDistinct("condition_event_key").alias("keys"),
).first()

grain_ok = (
    int(target_profile["rows"]) == prepared_conditions.count()
    and int(target_profile["rows"]) == int(target_profile["keys"])
)

final_checks = [
    ("source_grain", len(source_failures) == 0),
    ("foreign_keys", len(fk_failures) == 0),
    ("fact_grain", grain_ok),
    ("external_location",
     normalize_location(registered_table_location(spark, TABLES["fact_conditions"]))
     == normalize_location(fact_location)),
]

display(spark.createDataFrame(
    [(n, "PASS" if ok else "FAIL") for n, ok in final_checks],
    ["validation_area","status"],
))

failed = [n for n, ok in final_checks if not ok]
if run_validation and failed:
    raise RuntimeError("03 Fact Conditions failed: " + ", ".join(failed))

display(
    target.groupBy("condition_code","condition_description")
    .agg(
        F.count("*").alias("condition_events"),
        F.countDistinct("patient_key").alias("unique_patients"),
        F.sum(F.when(F.col("is_active_condition"),1).otherwise(0)).alias("active_events"),
        F.round(F.avg("condition_duration_days"),2).alias("avg_duration_days"),
    )
    .orderBy(F.desc("condition_events"))
    .limit(25)
)

print("LAB 06 EXTERNAL V2 — FACT CONDITIONS COMPLETE")
print("Serverless compatibility: PASS")
print("REFRESH TABLE calls: 0")
print("Next: lab06_04_aggregations")